In [1]:
import os
# 1. [关键] 必须设置缓存到数据盘 (50G硬盘保命设置)
os.environ["HF_HOME"] = "/root/autodl-tmp/hf_cache"
# 2. [关键] 关闭 HF_TRANSFER 加速 (解决 RuntimeError: no permits available)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
# 3. [关键] 关闭 Xet 加速 (解决 CAS service error)
os.environ["HF_HUB_DISABLE_XET"] = "1"
# 4. [关键] 使用国内镜像 (解决连接超时)
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import torch
from datasets import Dataset, load_dataset
import pandas as pd
from tqdm import tqdm
import random
from unsloth import FastLanguageModel, PatchDPOTrainer
from unsloth import is_bfloat16_supported
from trl import DPOTrainer, DPOConfig
import sys
sys.path.append("..")
from src.templates import SYSTEM_PROMPT, USER_PROMPT

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
model_path = "../outputs/qwen3_cot_finetuned_data_augmentation*0.7_LoraRank128"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

==((====))==  Unsloth 2025.11.1: Fast Qwen3 patching. Transformers: 4.57.2.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Unsloth 2025.11.1 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 4096, padding_idx=151654)
        (layers): ModuleList(
          (0-2): 3 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=64, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=64, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.

In [3]:
dataset = load_dataset("json", data_files="../data/processed/train_cot_cleaned_balanced_soft.jsonl", split="train")
train_data = dataset.to_pandas().to_dict(orient="records")
print(f"数据加载完成，共 {len(train_data)} 条。")
print(f"示例数据: {train_data[0]}")

dpo_data = []

数据加载完成，共 6920 条。
示例数据: {'text': 'The large selection of bruschettas, paninis, tramezzinis keep the palate from stagnating.', 'aspect': 'paninis', 'polarity': 'positive', 'target_output': '<think>\n1. Identify the target aspect: the word “paninis” appears in the list “bruschettas, paninis, tramezzinis.”  \n2. Examine the verb phrase that follows the list: “keep the palate from stagnating.”  \n3. The verb “keep” is used with a positive connotation here—it implies that the items (including paninis) actively prevent stagnation, suggesting a beneficial effect.  \n4. The sentence structure treats the entire list as a collective subject that performs the action of preventing stagnation.  \n5. Because the target aspect “paninis” is part of this collective subject, the positive action (“keep the palate from stagnating”) applies to it.  \n6. No negative or neutral modifiers are attached to “paninis”; there are no contrasting or contradictory phrases.  \n7. Therefore, the sentiment directed towar

In [4]:
batch_size = 64
for i in tqdm(range(0, len(train_data), batch_size)):
    batch = train_data[i : i+batch_size]
    
    # 构造 Prompts
    prompts = []
    for row in batch:
        user_content = USER_PROMPT.format(text=row['text'], aspect=row.get('term', row.get('aspect')))
        msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_content}]
        prompts.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))
        
    # 推理 (使用较高的 Temperature 诱导错误)
    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=True,      # 必须采样
            temperature=1.2,     # 高温！让模型更容易胡说八道
            top_p=0.95,
            use_cache=True
        )
        
    generated_texts = tokenizer.batch_decode(outputs[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)
    
    # 筛选 Rejected
    for row, gen_text, prompt in zip(batch, generated_texts, prompts):
        ground_truth = row['polarity'].lower()
        
        # 提取生成的 Label
        pred_label = "unknown"
        if "Final Sentiment:" in gen_text:
            pred_label = gen_text.split("Final Sentiment:")[-1].strip().lower().rstrip(".")
            
        # 逻辑判断：如果生成的标签和真实标签不一致，就是一个好的 Rejected
        # 特别关注：GT 是 Neutral 但模型生成了 Positive/Negative 的情况
        if pred_label != "unknown" and pred_label != ground_truth:
            dpo_data.append({
                "prompt": prompt,          # DPO 需要格式化好的 Prompt
                "chosen": row['target_output'], # 老师生成的完美答案
                "rejected": gen_text       # 模型生成的错误答案
            })
        else:
            # 如果模型太强没骗到，我们可以【人工构造】一个硬负例 (Hard Negative)
            # 策略：保留 Rationale，强行改错 Label
            # 这会告诉模型：光有推理不行，结论必须对
            fake_label = "positive" if ground_truth != "positive" else "negative"
            # 简单粗暴地把 target_output 里的 label 换掉
            fake_rejected = row['target_output'].replace(f"Final Sentiment: {ground_truth}", f"Final Sentiment: {fake_label}")
            # 注意：要确保 replace 成功（大小写匹配问题），如果没匹配上就跳过
            if fake_rejected != row['target_output']:
                dpo_data.append({
                    "prompt": prompt,
                    "chosen": row['target_output'],
                    "rejected": fake_rejected
                })

print(f"✅ DPO 数据集构建完成，共收集到 {len(dpo_data)} 条偏好对。")

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 109/109 [1:51:49<00:00, 61.55s/it]

✅ DPO 数据集构建完成，共收集到 6920 条偏好对。


In [5]:
# 保存
df_dpo = pd.DataFrame(dpo_data)
# 简单去重，防止 prompt 重复
df_dpo = df_dpo.drop_duplicates(subset=['prompt'])
df_dpo.to_json("dpo_dataset.jsonl", orient="records", lines=True)

In [6]:
# 1. 开启 DPO Patch
PatchDPOTrainer()

In [7]:
# 2. 加载 SFT 模型 (作为 Policy Model 也就是我们要训练的模型)
# 注意：这里加载的是你 SFT 后的模型
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,
    max_seq_length = 2048,
    load_in_4bit = True,
)

==((====))==  Unsloth 2025.11.1: Fast Qwen3 patching. Transformers: 4.57.2.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [8]:
# 3. 加载数据集
dataset = load_dataset("json", data_files="dpo_dataset.jsonl", split="train")

Generating train split: 0 examples [00:00, ? examples/s]

In [9]:
# 4. 配置 DPO Trainer
dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None, # Unsloth 优化：不需要加载 Ref Model，它会自动处理，省一半显存！
    tokenizer = tokenizer,
    train_dataset = dataset,
    beta = 0.1, # DPO 的核心超参，通常 0.1，如果训练不稳定可调大到 0.2
    max_prompt_length = 1024,
    max_length = 2048,
    args = DPOConfig(
        per_device_train_batch_size = 1, # DPO 显存占用大，建议从 1 开始试
        gradient_accumulation_steps = 8, # 以此来弥补 Batch Size
        warmup_ratio = 0.1,
        num_train_epochs = 1,            # DPO 很容易过拟合，通常 1 个 Epoch 就够了
        learning_rate = 5e-6,            # DPO 的学习率要比 SFT 低很多！(SFT是 2e-4，这里用 5e-6 或 1e-6)
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        output_dir = "../outputs/qwen3_cot_dpo",
        seed = 42,
    ),
)

Extracting prompt in train dataset (num_proc=64):   0%|          | 0/4730 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=64):   0%|          | 0/4730 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=64):   0%|          | 0/4730 [00:00<?, ? examples/s]

In [10]:
# 5. 开始训练
dpo_trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,730 | Num Epochs = 1 | Total steps = 592
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 174,587,904 of 8,365,323,264 (2.09% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected,eval_logits / chosen,eval_logits / rejected,nll_loss
10,0.728600,33.569454,31.953262,0.800000,1.616191,-49.748192,-72.359489,-3.541621,-3.338137,0,0,0
20,0.443000,36.205536,31.308023,0.912500,4.897509,-47.009705,-72.047745,-3.675493,-3.354101,No Log,No Log,No Log
30,0.340900,33.494801,29.468838,0.975000,4.025957,-45.647820,-83.265602,-3.650174,-3.370672,No Log,No Log,No Log
40,0.160400,33.167366,27.978546,0.975000,5.188816,-40.572670,-91.822899,-3.981502,-3.738354,No Log,No Log,No Log
50,0.171600,34.171822,28.358997,0.975000,5.812829,-41.804905,-102.920853,-4.025281,-3.835382,No Log,No Log,No Log
60,0.223300,35.706673,28.429083,0.975000,7.277584,-35.265228,-108.445473,-4.316283,-4.132915,No Log,No Log,No Log
70,0.154700,37.786736,28.578680,0.987500,9.208055,-32.229450,-111.213768,-4.454592,-4.135821,No Log,No Log,No Log
80,0.002900,36.127541,27.429043,1.000000,8.698493,-26.963404,-108.068726,-4.262573,-4.085476,No Log,No Log,No Log
90,0.558000,36.886177,30.569225,0.962500,6.316951,-24.458977,-113.865662,-4.283004,-4.171124,No Log,No Log,No Log
100,0.003600,35.234642,27.449871,1.000000,7.784770,-23.268255,-108.906372,-4.338469,-4.306929,No Log,No Log,No Log


TrainOutput(global_step=592, training_loss=0.08724349347844632, metrics={'train_runtime': 2510.9997, 'train_samples_per_second': 1.884, 'train_steps_per_second': 0.236, 'total_flos': 0.0, 'train_loss': 0.08724349347844632, 'epoch': 1.0})

In [ ]:
# 6. 保存
model.save_pretrained("../outputs/qwen3_cot_dpo_final")
tokenizer.save_pretrained("../outputs/qwen3_cot_dpo_final")